In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
sandbox_df = pd.read_csv('../data/processed/august_2024_uk_sandbox.csv')

In [ ]:
sandbox_df.head()

Initial integrity check

- 10,249 posts for one month
- num_comments and score are int - good
- 6048 missing values in selftext - the column that contains the contents of the post. Bad - 59%

In [ ]:
# check num rows/cols
sandbox_df.shape

In [ ]:
# check data types of cols
sandbox_df.dtypes

In [ ]:
# null values check
null_vals = sandbox_df.isnull().sum()
null_vals

In [ ]:
# proportion null
null_vals/sandbox_df.shape[0]

Class balances

In [ ]:
# check subreddit counts
subr_counts = sandbox_df['subreddit'].value_counts()
subr_counts

In [ ]:
# check for subreddit imbalances
subr_counts/sandbox_df.shape[0]

Stats for virality

In [ ]:
# descriptive stats
virality_cols_df = sandbox_df.loc[:, ['num_comments', 'score']]

virality_cols_df.describe()

In [ ]:
# Look at distribution of scores
plt.hist(virality_cols_df.loc[:, 'score'], bins=50)
plt.xlabel('score')
plt.ylabel('num_posts')
plt.show()

Vast majority of reddit posts have a score of <100. Big positive skew. Likely to need log transformed score for models

NB - score = num up votes - num down votes

In [ ]:
# View top 10 highest scoring posts of the month 

sorted_by_score = sandbox_df.sort_values(by=['score'], ascending=False)
sorted_by_score.iloc[:10, :]

## Step 4: Text Profile & Length Analysis
This step sets the stage for Pipeline A (The BERT/NLP Classifier). Text models are highly sensitive to document length.

Token/Word Counting: Create a new metric calculating the word count or character count for each post title and body text.

Length Statistics: Find the average length of a post. Are titles short and punchy? Are body texts massive essays? BERT models typically have a maximum token limit (usually 512 tokens), so you need to check if your data contains text blocks that will eventually get cut off.

In [ ]:
sandbox_df['title_length'] = sandbox_df.loc[:, 'title'].str.split().str.len()
sandbox_df['body_length'] = sandbox_df.loc[:, 'selftext'].str.split().str.len()
sandbox_df

In [ ]:
sandbox_df.loc[:, ['title_length', 'body_length']].describe()

## Step 5: Domain and Outward-Link Profiling
This is the ultimate check to prepare for your Snorkel Distant Supervision rules.

Domain Extraction: Extract the core base domains (e.g., bbc.co.uk, theguardian.com) from your raw url string column.

Top Outlet Ranking: Find the top 20 most frequently linked domains across the entire dataset.

Coverage Verification: Cross-reference this list with the balanced media checklist we made earlier. Do you see a healthy presence of mainstream broadsheets, left/right tabloids, and alternative media? This tells you exactly how active your upcoming labeling functions will be.

In [ ]:
core_domains = sandbox_df.loc[:, 'url'].str.split('//').str[1].str.split('/').str[0]

In [ ]:
core_domains.value_counts()